# fasthtml

> In-kernel live inspector panel served via FastHTML

In [ ]:
#| default_exp fasthtml

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import json
from urllib.parse import quote
from fasthtml.common import to_xml
from starlette.testclient import TestClient
import paar.bridge as B, paar.fasthtml as F
from paar.fasthtml import app, _node, _broadcast, _acc
from paar.core import VarInfo

class _IP:
    def __init__(self, ns): self.user_ns=ns; self.user_ns_hidden=set()
    class events:
        @staticmethod
        def register(n, fn): pass

def test_node_scalar_no_toggle():
    html = to_xml(_node(VarInfo(name='n', type='int', value='5', accessor=('n',), path='n')))
    assert 'n' in html and 'int' in html and '5' in html
    assert '/expand' not in html and '<details' not in html
def test_node_container_details():
    html = to_xml(_node(VarInfo(name='d', type='dict', value='{...}',
                                is_container=True, accessor=('d',), path='d')))
    assert '<details' in html and '<summary' in html
    assert 'paar-children' in html and '/expand?accessor=' in html
def test_node_container_lazy_once():
    html = to_xml(_node(VarInfo(name='d', type='dict', is_container=True, accessor=('d',), path='d')))
    assert 'once' in html   # hx-trigger="click once" -> children fetched only on first open
def test_node_more_sentinel():
    html = to_xml(_node(VarInfo(name='…', type='', value='150 more', accessor=('L',), path='L', more_offset=100)))
    assert '/expand?accessor=' in html and 'offset=100' in html and '150 more' in html
    assert 'hx-target="this"' in html and '<details' not in html   # clicking swaps itself for the next page
def test_rows_route_lists_vars():
    B.get_ipython = lambda: _IP({'alpha': 123})
    r = TestClient(app).get('/rows?profile=standard')
    assert r.status_code==200 and 'alpha' in r.text and '123' in r.text and 'id="rows"' in r.text
def test_home_wires_ws_and_loader():
    r = TestClient(app).get('/')
    assert r.status_code==200
    assert 'ws-connect="/live"' in r.text and 'hx-ext="ws"' in r.text
    assert 'id="rows"' in r.text and '/rows' in r.text
def test_home_has_profile_select():
    r = TestClient(app).get('/')
    assert '<select' in r.text and 'name="profile"' in r.text
    assert 'minimal' in r.text and 'standard' in r.text   # profile options rendered
def test_rows_profile_filters_and_groups():
    import math
    B.get_ipython = lambda: _IP({'myvar': 5, 'mymod': math})
    mn = TestClient(app).get('/rows?profile=minimal').text
    assert 'myvar' in mn and 'Modules' not in mn          # minimal hides the module group
    st = TestClient(app).get('/rows?profile=standard').text
    assert 'myvar' in st and 'Modules' in st and 'mymod' in st   # standard groups modules
def test_expand_route_returns_children():
    B.get_ipython = lambda: _IP({'d': {'x': 1}})
    r = TestClient(app).get('/expand?accessor=' + quote(json.dumps(['d'])))
    assert r.status_code==200 and 'x' in r.text and '1' in r.text
def test_expand_route_paging():
    B.get_ipython = lambda: _IP({'L': list(range(250))})
    r = TestClient(app).get('/expand?accessor=' + quote(json.dumps(['L'])) + '&offset=100')
    assert r.status_code==200 and '100' in r.text and 'offset=200' in r.text   # trailing load-more points at page 3
def test_broadcast_drops_dead_clients():
    F._clients.clear()
    def _bad_send(_): raise RuntimeError('dead')
    F._clients.append(('not-a-loop', _bad_send))
    _broadcast('x')
    assert F._clients==[]
def test_node_grid_only_toggles_grid():
    html = to_xml(_node(VarInfo(name='arr', type='ndarray', is_grid=True, accessor=('arr',), path='arr')))
    assert '<details' in html and '/grid?accessor=' in html and 'paar-gridbox' in html
    assert '/expand' not in html   # not a container -> the row toggles the grid, no tree
def test_node_grid_and_container_shows_both():
    html = to_xml(_node(VarInfo(name='arr', type='ndarray', is_grid=True, is_container=True,
                                accessor=('arr',), path='arr')))
    assert '/expand?accessor=' in html and '/grid?accessor=' in html   # tree + grid coexist
    assert 'paar-gridbox' in html and 'paar-children' in html and 'paar-gridtoggle' in html
    assert html.count('<details') >= 2 and html.count('click once') >= 2   # both are lazy toggles
def test_grid_route_renders_table():
    import numpy as np
    B.get_ipython = lambda: _IP({'arr': np.arange(6).reshape(2,3)})
    r = TestClient(app).get('/grid?accessor=' + quote(json.dumps(['arr'])) + '&roff=0&coff=0')
    assert r.status_code==200 and '<table' in r.text and '5' in r.text  # last cell value present
def test_exec_route_assignment():
    import paar.runtime as R, types
    class _Sh:
        def __init__(s): s.user_ns={}
        def run_cell(s, code, store_history=True):
            exec(code, s.user_ns)
            return types.SimpleNamespace(result=None, error_in_exec=None, error_before_exec=None, success=True)
    sh = _Sh(); R.get_ipython = lambda: sh
    r = TestClient(app).post('/exec', data={'code':'q = 5'})
    assert r.status_code == 200 and sh.user_ns['q'] == 5 and 'exec-out' in r.text
def test_exec_bar_renders():
    r = TestClient(app).get('/')
    assert 'hx-post="/exec"' in r.text and 'id="paar-code"' in r.text
    assert 'name="code"' in r.text and 'isolated' in r.text   # hidden textarea still named code
def test_editor_script_present():
    r = TestClient(app).get('/')
    assert 'codejar' in r.text.lower() and 'shiftKey' in r.text   # editor loaded + shift+enter binding
def test_edit_route_returns_form():
    r = TestClient(app).post('/edit', data={'accessor': json.dumps(['x'])})
    assert r.status_code == 200 and 'hx-post="/set"' in r.text and 'name="expr"' in r.text
def test_set_route_writes_and_refreshes():
    import paar.runtime as R
    class _IP2:
        def __init__(s): s.user_ns={'x':1}; s.user_ns_hidden=set()
        class events:
            @staticmethod
            def register(n, fn): pass
    ip = _IP2(); B.get_ipython = lambda: ip; R.get_ipython = lambda: ip
    F._clients.clear()
    r = TestClient(app).post('/set', data={'accessor': json.dumps(['x']), 'expr':'42'})
    assert r.status_code == 200 and ip.user_ns['x'] == 42 and 'id="rows"' in r.text
def test_set_route_error_inline():
    import paar.runtime as R
    class _IP3:
        def __init__(s): s.user_ns={'s':{1,2}}; s.user_ns_hidden=set()
        class events:
            @staticmethod
            def register(n, fn): pass
    ip = _IP3(); B.get_ipython = lambda: ip; R.get_ipython = lambda: ip
    r = TestClient(app).post('/set', data={'accessor': json.dumps(['s',0]), 'expr':'9'})
    assert r.status_code == 200 and 'exec-out' in r.text   # error surfaced OOB, tree intact
def test_node_value_is_editable():
    html = to_xml(_node(VarInfo(name='x', type='int', value='1', accessor=('x',), path='x')))
    assert '/edit?accessor=' in html and 'paar-val' in html
def test_nodes_have_data_attrs():
    html = to_xml(_node(VarInfo(name='arr', type='ndarray', value='x', accessor=('arr',), path='arr')))
    assert 'data-name="arr"' in html and 'data-type="ndarray"' in html
def test_filter_bar_renders():
    r = TestClient(app).get('/')
    assert 'paar-filter-name' in r.text and 'paar-filter-type' in r.text
def test_exec_result_not_editable():
    html = to_xml(_node(VarInfo(name='result', type='int', value='7', accessor=('_',), path='_')))
    assert 'paar-val' not in html and '/edit?accessor=' not in html   # `_` result row is not click-to-edit
def test_complete_route_json():
    import paar.runtime as R
    class _Sh:
        def __init__(s): s.user_ns={'alpha':1,'alto':2}
        def complete(s, text=None, line=None, cursor_pos=None):
            import re
            q = text if text is not None else (line or '')[:cursor_pos]
            tok = re.split(r'[^\w.]', q)[-1]
            return tok, [k for k in s.user_ns if k.startswith(tok)]
    R.get_ipython=lambda: _Sh()
    r = TestClient(app).get('/complete?code=al&pos=2')
    assert r.status_code==200
    j = r.json(); assert j['from']==0 and set(j['matches'])=={'alpha','alto'}
def test_editor_has_autocomplete_and_clear():
    r = TestClient(app).get('/')
    assert '/complete?code=' in r.text                 # editor fetches completions
    assert 'htmx:afterRequest' in r.text               # clear-on-run wired
    assert 'paar-ac' in r.text                          # completion popup styling present
def test_sessions_and_session_routes():
    import paar.runtime as R, tempfile, types
    from pathlib import Path
    R.SESSION_DIR = Path(tempfile.mkdtemp())
    class _Sh:
        def __init__(s): s.user_ns={}
        def run_cell(s, code, store_history=True):
            exec(code, s.user_ns)
            return types.SimpleNamespace(result=None, error_in_exec=None, error_before_exec=None, success=True)
    R.get_ipython=lambda: _Sh()
    R.run('zz = 5')
    assert 'session_' in TestClient(app).get('/sessions').text
    name = R.list_sessions()[0][0]
    s2 = TestClient(app).get('/session?name='+name).text
    assert 'zz = 5' in s2 and 'paar-reuse' in s2
def test_home_has_sessions_panel():
    r = TestClient(app).get('/')
    assert 'Sessions' in r.text and '/sessions' in r.text
def test_editor_exposes_paarload():
    r = TestClient(app).get('/')
    assert 'paarLoad' in r.text and 'paar-reuse' in r.text
def test_sessions_lists_current():
    r = TestClient(app).get('/sessions')
    assert 'current' in r.text   # current session always shown at top
for t in [test_node_scalar_no_toggle,test_node_container_details,test_node_container_lazy_once,
          test_node_more_sentinel,test_rows_route_lists_vars,test_home_wires_ws_and_loader,
          test_home_has_profile_select,test_rows_profile_filters_and_groups,
          test_expand_route_returns_children,test_expand_route_paging,
          test_broadcast_drops_dead_clients,test_node_grid_only_toggles_grid,
          test_node_grid_and_container_shows_both,test_grid_route_renders_table,
          test_exec_route_assignment,test_exec_bar_renders,test_editor_script_present,
          test_edit_route_returns_form,test_set_route_writes_and_refreshes,
          test_set_route_error_inline,test_node_value_is_editable,
          test_nodes_have_data_attrs,test_filter_bar_renders,
          test_exec_result_not_editable,
          test_complete_route_json,test_editor_has_autocomplete_and_clear,
          test_sessions_and_session_routes,test_home_has_sessions_panel,
          test_editor_exposes_paarload,test_sessions_lists_current]: t()

In [ ]:
#| export
import asyncio, threading, json
from pathlib import Path
from urllib.parse import quote
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi, HTMX
from paar.bridge import Bridge, on_change
from paar.core import VarInfo
from paar.snapshot import PROFILES
from paar.runtime import run, complete, list_sessions, read_session, current_session
from starlette.responses import JSONResponse

In [ ]:
#| export
_hljs = (   # inline python syntax-highlighting; theme follows the OS/browser color scheme
    Link(rel='stylesheet', media='(prefers-color-scheme: dark)',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css'),
    Link(rel='stylesheet', media='(prefers-color-scheme: light)',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-light.min.css'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/highlight.min.js'),
    Script("htmx.onLoad(el=>el.querySelectorAll('code.pv:not([data-highlighted])')"
           ".forEach(c=>hljs.highlightElement(c)))"),
    Script("""
function paarFilter(){
  const nq=(document.querySelector('.paar-filter-name')?.value||'').toLowerCase();
  const tq=document.querySelector('.paar-filter-type')?.value||'';
  document.querySelectorAll('#rows .paar-node[data-name]').forEach(n=>{
    const nm=(n.getAttribute('data-name')||'').toLowerCase();
    const ty=n.getAttribute('data-type')||'';
    const show=(!nq||nm.includes(nq))&&(!tq||ty===tq);
    n.style.display=show?'':'none';
    if(show){const g=n.closest('details.paar-group'); if(g)g.open=true;}
  });
}
function paarSyncTypes(){
  const sel=document.querySelector('.paar-filter-type'); if(!sel)return;
  const cur=sel.value;
  const types=[...new Set([...document.querySelectorAll('#rows .paar-node[data-type]')]
    .map(n=>n.getAttribute('data-type')).filter(Boolean))].sort();
  sel.innerHTML='<option value="">all types</option>'+types.map(t=>'<option value="'+t+'">'+t+'</option>').join('');
  sel.value=cur; paarFilter();
}
htmx.onLoad(()=>{ if(window.paarSyncTypes) paarSyncTypes(); });
"""),
    Script("""
import {CodeJar} from 'https://cdn.jsdelivr.net/npm/codejar@4.2.0/dist/codejar.js';
function paarCaret(el){
  const s=window.getSelection(); if(!s.rangeCount) return 0;
  const r=s.getRangeAt(0).cloneRange(); const pre=r.cloneRange();
  pre.selectNodeContents(el); pre.setEnd(r.endContainer, r.endOffset);
  return pre.toString().length;
}
function paarSetCaret(el, off){
  const w=document.createTreeWalker(el, NodeFilter.SHOW_TEXT); let n, cur=0;
  while(n=w.nextNode()){ const l=n.nodeValue.length;
    if(cur+l>=off){ const r=document.createRange(); r.setStart(n, off-cur); r.collapse(true);
      const s=window.getSelection(); s.removeAllRanges(); s.addRange(r); return; } cur+=l; }
}
function paarInitEditor(){
  const el=document.querySelector('#paar-code'); if(!el||el.dataset.cj) return; el.dataset.cj='1';
  const src=document.querySelector('#paar-code-src');
  const form=document.querySelector('#paar-exec-form');
  const hl=e=>{ e.innerHTML=hljs.highlight(e.textContent,{language:'python'}).value; };
  const jar=CodeJar(el, hl, {tab:'    '});
  jar.updateCode(src.value||''); jar.onUpdate(c=>{ src.value=c; });
  window.paarLoad=c=>{ jar.updateCode(c); src.value=c; el.focus(); };
  form.addEventListener('htmx:afterRequest', e=>{ if(e.detail.successful){ jar.updateCode(''); src.value=''; } });
  // ---- autocomplete popup ----
  let items=[], sel=0, from=0, box=null;
  const esc=s=>String(s).replace(/[&<>]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;'}[c]));
  const close=()=>{ if(box){box.remove(); box=null;} items=[]; };
  const render=()=>{
    if(!box){ box=document.createElement('div'); box.className='paar-ac'; document.body.appendChild(box); }
    box.innerHTML=items.map((m,i)=>'<div class="paar-ac-item'+(i===sel?' sel':'')+'">'+esc(m)+'</div>').join('');
    Array.from(box.children).forEach((c,i)=>c.onmousedown=ev=>{ ev.preventDefault(); accept(i); });
    const sr=window.getSelection(); const rects=sr.rangeCount?sr.getRangeAt(0).getClientRects():[];
    const rc=rects.length?rects[0]:el.getBoundingClientRect();
    box.style.left=rc.left+'px'; box.style.top=(rc.bottom+2)+'px';
    const selEl=box.children[sel]; if(selEl) selEl.scrollIntoView({block:'nearest'});
  };
  const accept=i=>{ const m=items[i]; if(m==null) return; const code=src.value, pos=paarCaret(el);
    const nc=code.slice(0,from)+m+code.slice(pos); jar.updateCode(nc); src.value=nc; paarSetCaret(el, from+m.length); close(); };
  const trigger=async()=>{ const code=src.value, pos=paarCaret(el);
    let res; try{ res=await fetch('/complete?code='+encodeURIComponent(code)+'&pos='+pos).then(r=>r.json()); }catch(e){ return; }
    if(!res.matches||!res.matches.length){ close(); return; } items=res.matches; sel=0; from=res['from']; render(); };
  el.addEventListener('keydown', e=>{
    if(box){
      if(e.key==='ArrowDown'){e.preventDefault(); e.stopImmediatePropagation(); sel=(sel+1)%items.length; render(); return;}
      if(e.key==='ArrowUp'){e.preventDefault(); e.stopImmediatePropagation(); sel=(sel-1+items.length)%items.length; render(); return;}
      if(e.key==='Enter'||e.key==='Tab'){e.preventDefault(); e.stopImmediatePropagation(); accept(sel); return;}
      if(e.key==='Escape'){e.preventDefault(); close(); return;}
    }
    if(e.key==='Enter' && e.shiftKey){ e.preventDefault(); e.stopImmediatePropagation(); htmx.trigger('#paar-exec-form','submit'); return; }
    if(e.key===' ' && e.ctrlKey){ e.preventDefault(); e.stopImmediatePropagation(); trigger(); return; }
  }, true);
  el.addEventListener('keyup', e=>{
    if(e.ctrlKey||e.metaKey||e.altKey) return;
    if(['ArrowDown','ArrowUp','ArrowLeft','ArrowRight','Enter','Tab','Escape',' '].includes(e.key)) return;
    if(/^[\\w.]$/.test(e.key)){ clearTimeout(el._acT); el._acT=setTimeout(trigger,120); } else close();
  }, false);
  el.addEventListener('blur', ()=>setTimeout(close,150));
  document.addEventListener('click', e=>{ const b=e.target.closest('.paar-reuse'); if(!b) return;
    const cc=b.parentElement.querySelector('code'); if(cc&&window.paarLoad) window.paarLoad(cc.textContent); });
}
if(document.readyState!=='loading') paarInitEditor();
else document.addEventListener('DOMContentLoaded', paarInitEditor);
""", type='module'))
_css = Style(
    ':root{--pico-primary:#0172ad;--pico-muted-border-color:#cfd6dd;--pico-background-color:#fff;--paar-fg:#1b1f24} '
    '@media (prefers-color-scheme:dark){:root{--pico-primary:#5aa7e6;--pico-muted-border-color:#343b43;--pico-background-color:#0d1117;--paar-fg:#c9d1d9}} '
    'body{color:var(--paar-fg);background:var(--pico-background-color);margin:0} '
    '.paar-node{font-size:.8rem;line-height:1.45;'
    'font-family:ui-monospace,SFMono-Regular,Menlo,monospace} '
    '.paar-children{margin-left:.55rem;padding-left:.4rem;'
    'border-left:1px solid var(--pico-muted-border-color)} '
    'details.paar-node{margin:0} '
    'summary{cursor:pointer;list-style:none;display:block;margin:0;padding:0} '
    'summary::-webkit-details-marker{display:none} '
    'summary::before{content:"\\25B8";display:inline-block;width:1em;opacity:.45} '
    'details[open]>summary::before{content:"\\25BE"} '
    '.paar-leaf{padding-left:1em} '
    '.paar-name{color:var(--pico-primary)} '
    '.paar-eq{opacity:.4} '
    '.paar-type{opacity:.5;font-size:.72rem} '
    '.paar-node code{white-space:pre;background:none;padding:0} '
    'code.hljs{display:inline;background:none;padding:0} '
    '.paar-group>summary{font-weight:600;opacity:.65;text-transform:uppercase;'
    'font-size:.7rem;letter-spacing:.03em} '
    '.paar-profile{font:inherit;font-size:.75rem;padding:.1rem .4rem;margin:0 0 .4rem;width:auto;'
    'border:1px solid var(--pico-muted-border-color);border-radius:5px;background:var(--pico-background-color);color:inherit} '
    '.paar-gridbox:not(:empty){margin:.25rem 0 .25rem .85rem} '
    '.paar-grid{max-height:360px;overflow:auto} '
    '.paar-grid table{font-size:.8rem;margin:0} '
    '.paar-grid th{position:sticky;top:0;background:var(--pico-background-color)} '
    '.paar-gridnav{margin:.25rem 0} '
    '.paar-page{margin-right:.5rem;cursor:pointer} '
    '.paar-grid-label{opacity:.6;cursor:pointer} '
    '.paar-exec{display:flex;flex-direction:column;gap:.35rem;margin:0 0 .5rem} '
    '.paar-exec-controls{display:flex;gap:.5rem;align-items:center;justify-content:flex-end} '
    '.paar-code{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:.8rem;line-height:1.45;'
    'min-height:2em;max-height:14em;overflow:auto;white-space:pre;tab-size:4;'
    'border:1px solid var(--pico-muted-border-color);border-radius:6px;padding:.35rem .5rem;'
    'background:var(--pico-background-color)} '
    '.paar-code:focus{outline:none;border-color:var(--pico-primary)} '
    '.paar-exec button{font:inherit;font-size:.8rem;padding:.2rem .8rem;border-radius:6px;cursor:pointer;'
    'border:1px solid var(--pico-primary);background:var(--pico-primary);color:#fff} '
    '.paar-exec button:hover{opacity:.9} '
    '.paar-exec label{display:flex;align-items:center;gap:.3rem;font-size:.75rem} '
    '.paar-out{font-size:.75rem;white-space:pre-wrap;margin:.2rem 0} '
    '.paar-val{cursor:pointer;border-bottom:1px dotted var(--pico-muted-border-color)} '
    '.paar-edit-input{font:inherit;font-size:.8rem;padding:.15rem .4rem;margin:0;'
    'border:1px solid var(--pico-muted-border-color);border-radius:5px;background:var(--pico-background-color);color:inherit} '
    '.paar-filter{display:flex;gap:.4rem;align-items:center;margin:0 0 .4rem} '
    '.paar-filter input,.paar-filter select{font:inherit;font-size:.8rem;padding:.15rem .4rem;margin:0;'
    'border:1px solid var(--pico-muted-border-color);border-radius:5px;background:var(--pico-background-color);color:inherit} '
    '.paar-filter-name{flex:1;min-width:0} '
    '.paar-filter-type{width:auto} '
    '.paar-ac{position:fixed;z-index:9999;max-height:14em;overflow:auto;min-width:8em;'
    'background:var(--pico-background-color);border:1px solid var(--pico-muted-border-color);'
    'border-radius:6px;font:.8rem ui-monospace,SFMono-Regular,Menlo,monospace;box-shadow:0 4px 14px rgba(0,0,0,.18)} '
    '.paar-ac-item{padding:.1rem .5rem;cursor:pointer;white-space:pre} '
    '.paar-ac-item.sel{background:var(--pico-primary);color:#fff} '
    '.paar-session-cells{margin-left:.6rem} '
    '.paar-session-cell{display:flex;gap:.4rem;align-items:flex-start;margin:.15rem 0} '
    '.paar-session-cell code{white-space:pre-wrap;font-size:.78rem} '
    '.paar-reuse{flex:0 0 auto;cursor:pointer;font-size:.75rem;padding:0 .35rem;border-radius:5px;'
    'border:1px solid var(--pico-muted-border-color);background:var(--pico-background-color);color:inherit} ')
bridge = Bridge()
app,rt = fast_app(exts='ws', pico=False, hdrs=(_css, *_hljs))   # ws ext + hljs
_clients = []   # list[(loop, send)]
_clients_lock = threading.Lock()
_server = None
_profile = 'standard'   # active inspector profile (mutated by the /rows dropdown)

In [ ]:
#| export
def _acc(accessor):
    "URL-safe encoding of a positional accessor."
    return quote(json.dumps(list(accessor)), safe='')

def _head(v:VarInfo):
    "PyCharm-style row: name = {type: shape} value; value is click-to-edit when it has an accessor."
    typ = f'{v.type}: {v.shape}' if v.shape else v.type
    val = Code(v.value, cls='pv language-python')
    if v.accessor and v.accessor != ('_',) and v.more_offset is None and not v.is_error:
        val = Span(val, cls='paar-val', title='click to edit',
                   hx_post=f'/edit?accessor={_acc(v.accessor)}',
                   hx_target='this', hx_swap='innerHTML', hx_trigger='click')
    return Span(Span(v.name, cls='paar-name'), Span(' = ', cls='paar-eq'),
                Span('{', typ, '}', cls='paar-type', title=v.qualifier), ' ',
                val,
                Small(f' {v.dtype}', cls='paar-type') if v.dtype else '',
                cls=('error' if v.is_error else None))

def _grid_toggle(v):
    "A collapsible '▦ grid' sub-node that lazily loads the paged table on first open."
    return Details(
        Summary(Span('▦ grid', cls='paar-grid-label'),
                hx_get=f'/grid?accessor={_acc(v.accessor)}&roff=0&coff=0',
                hx_target='next .paar-gridbox', hx_swap='innerHTML', hx_trigger='click once'),
        Div(cls='paar-gridbox'), cls='paar-node paar-gridtoggle')

def _more(v:VarInfo):
    "The load-more sentinel: clicking it swaps itself for the next page of children."
    return Div(Span(f'… {v.value}', cls='paar-grid-label'),
               hx_get=f'/expand?accessor={_acc(v.accessor)}&offset={v.more_offset}',
               hx_target='this', hx_swap='outerHTML', hx_trigger='click',
               cls='paar-node paar-leaf paar-more-row')

def _node(v:VarInfo):
    "Render a tree node: the load-more sentinel fetches the next page; containers expand; gridables also get a collapsible grid; scalars are plain."
    if v.more_offset is not None: return _more(v)
    head = _head(v)
    data = dict(data_name=v.name, data_type=v.type)
    if v.is_container:
        body = [_grid_toggle(v)] if v.is_grid else []   # grid toggle sits above the tree children
        return Details(
            Summary(head, hx_get=f'/expand?accessor={_acc(v.accessor)}',
                    hx_target='next .paar-children', hx_swap='innerHTML', hx_trigger='click once'),
            *body,
            Div(cls='paar-children'), cls='paar-node', **data)
    if v.is_grid:   # gridable but not otherwise expandable: the row itself toggles the grid
        return Details(
            Summary(head, ' ', Span('▦ grid', cls='paar-grid-label'),
                    hx_get=f'/grid?accessor={_acc(v.accessor)}&roff=0&coff=0',
                    hx_target='next .paar-gridbox', hx_swap='innerHTML', hx_trigger='click once'),
            Div(cls='paar-gridbox'), cls='paar-node', **data)
    return Div(head, cls='paar-node paar-leaf', **data)

def _filter_bar():
    "Name substring input + type dropdown; both filter the tree client-side."
    return Div(Input(name='fname', placeholder='filter name…', cls='paar-filter-name', oninput='paarFilter()'),
               Select(Option('all types', value=''), name='ftype', cls='paar-filter-type', onchange='paarFilter()'),
               cls='paar-filter')

def _grid_nav(page, acc):
    "Prev/next paging controls + window info for a grid page."
    roff, coff = page['roff'], page['coff']
    npr, npc = len(page['index']), len(page['headers'])
    nrows, ncols = page['nrows'], page['ncols']
    pr, pc = page['rows'], page['cols']
    def lnk(label, ro, co):
        return A(label, hx_get=f'/grid?accessor={_acc(acc)}&roff={ro}&coff={co}',
                 hx_target='closest .paar-gridbox', hx_swap='innerHTML', cls='paar-page')
    ctrls = []
    if roff > 0: ctrls.append(lnk('◀ rows', max(0, roff-pr), coff))
    if roff+npr < nrows: ctrls.append(lnk('rows ▶', roff+npr, coff))
    if coff > 0: ctrls.append(lnk('◀ cols', roff, max(0, coff-pc)))
    if coff+npc < ncols: ctrls.append(lnk('cols ▶', roff, coff+npc))
    info = Small(f'rows {roff}-{roff+npr} of {nrows}, cols {coff}-{coff+npc} of {ncols}')
    return Div(info, ' ', *ctrls, cls='paar-gridnav')

def _grid(page, acc):
    "Render a grid page as a scrollable table with paging controls."
    if page is None: return Div('not gridable')
    hdr = Tr(Th(''), *[Th(h) for h in page['headers']])
    body = [Tr(Th(ix), *[Td(c) for c in row]) for ix, row in zip(page['index'], page['cells'])]
    return Div(Div(Table(Thead(hdr), Tbody(*body)), cls='paar-grid'),
               _grid_nav(page, acc))

def _group(label, vs):
    "A collapsed category group (Special Variables / Functions / Modules) holding its member nodes."
    return Details(Summary(Span(label)),
                   Div(*[_node(v) for v in vs], cls='paar-children'),
                   cls='paar-node paar-group')

def _profile_select():
    "The profile dropdown; on change GETs /rows for the chosen profile."
    return Select(*[Option(p, value=p, selected=(p==_profile)) for p in PROFILES],
                  name='profile', hx_get='/rows', hx_target='#rows', hx_swap='outerHTML',
                  hx_trigger='change', cls='paar-profile')

def _rows_div():
    "The active profile's grouped node tree, wrapped in the #rows target."
    nodes = []
    for label, vs in bridge.view(_profile):
        if label is None: nodes += [_node(v) for v in vs]
        else: nodes.append(_group(label, vs))
    return Div(*nodes, id='rows')

def _loader(oob=False):
    "A #rows div that immediately GETs /rows (used initially and as the WS nudge)."
    kw = dict(hx_get='/rows', hx_trigger='load', hx_swap='outerHTML')
    if oob: kw['hx_swap_oob']='true'
    return Div(id='rows', **kw)

def _exec_bar():
    "Multi-line code editor (CodeJar + hljs); Shift+Enter runs, Enter=newline; isolated toggle."
    return Form(
        Div(id='paar-code', cls='paar-code hljs language-python'),
        Textarea(name='code', id='paar-code-src', style='display:none'),
        Div(Label(Input(type='checkbox', name='scope', value='isolated'), ' isolated'),
            Button('run', type='submit'), cls='paar-exec-controls'),
        hx_post='/exec', hx_target='#exec-out', hx_swap='outerHTML',
        id='paar-exec-form', cls='paar-exec')

def _exec_out(r):
    "Render an ExecResult as the #exec-out container: result row (if any) + stdout/error panels."
    parts = []
    if r.result is not None: parts.append(_node(r.result))
    if r.stdout: parts.append(Pre(r.stdout, cls='paar-out'))
    if r.error:  parts.append(Pre(r.error, cls='paar-out error'))
    return Div(*parts, id='exec-out')

In [ ]:
#| export
@rt('/')
def home():
    return Titled('paar', _profile_select(), _filter_bar(), _exec_bar(), Div(id='exec-out'),
                  Div(_loader(), hx_ext='ws', ws_connect='/live', id='paar'),
                  _sessions_panel())

@rt('/rows')
def rows(profile:str=None):
    global _profile
    if profile in PROFILES: _profile = profile
    return _rows_div()

@rt('/expand')
def expand_route(accessor:str, offset:int=0):
    # flat fragment: innerHTML on first open, or outerHTML replacing the load-more sentinel on a page click
    return tuple(_node(v) for v in bridge.expand(tuple(json.loads(accessor)), offset))

@rt('/grid')
def grid_route(accessor:str, roff:int=0, coff:int=0):
    acc = tuple(json.loads(accessor))
    return _grid(bridge.grid(acc, roff, coff), acc)

@rt('/exec')
def exec_route(code:str, scope:str='global'):
    return _exec_out(run(code, scope if scope in ('global','isolated') else 'global'))

@rt('/complete')
def complete_route(code:str='', pos:int=0):
    return JSONResponse(complete(code, pos))

@rt('/edit')
def edit_route(accessor:str):
    "Return an inline edit form for the value at `accessor`."
    return Form(Input(name='expr', placeholder='new value', cls='paar-edit-input', autofocus=True),
                Input(type='hidden', name='accessor', value=accessor),
                hx_post='/set', hx_target='#rows', hx_swap='outerHTML', cls='paar-edit')

@rt('/set')
def set_route(accessor:str, expr:str):
    "Write `expr` into the namespace at `accessor`; refresh the tree, or surface an inline error."
    err = bridge.set_value(tuple(json.loads(accessor)), expr)
    if err:
        return _rows_div(), Div(err, id='exec-out', cls='paar-out error', hx_swap_oob='true')
    _broadcast(to_xml(_loader(oob=True)))
    return _rows_div()

def _session_row(stem, label, n, once=True):
    return Details(Summary(f'{label} · {n}', hx_get=f'/session?name={stem}',
                           hx_target='next .paar-session-cells', hx_swap='innerHTML',
                           hx_trigger='click once' if once else 'click'),
                   Div(cls='paar-session-cells'), cls='paar-node')

def _session_rows():
    cur = current_session(); counts = dict(list_sessions())
    rows = [_session_row(cur, f'current · {Path.cwd().name}', counts.get(cur, 0), once=False)]
    rows += [_session_row(s, s, n) for s,n in list_sessions() if s != cur]
    return rows

@rt('/sessions')
def sessions_route(): return Div(*_session_rows())

@rt('/session')
def session_route(name:str=''):
    cells = read_session(name)
    if not cells: return Div('empty', cls='paar-type')
    return Div(*[Div(Button('↺', cls='paar-reuse', title='load into editor'),
                     Code(c, cls='pv language-python'), cls='paar-session-cell') for c in cells])

def _sessions_panel():
    "Bottom-of-inspector list of session notebooks; refreshes on open; current session first."
    return Details(Summary('Sessions', hx_get='/sessions', hx_target='next .paar-children',
                           hx_swap='innerHTML', hx_trigger='click'),
                   Div(cls='paar-children'), cls='paar-node paar-group', id='sessions')

def _drop(send):
    "Remove a client by its send identity (thread-safe)."
    with _clients_lock:
        _clients[:] = [(l,s) for (l,s) in _clients if s is not send]

async def _conn(send):
    with _clients_lock: _clients.append((asyncio.get_running_loop(), send))
async def _disconn(send): _drop(send)

@app.ws('/live', conn=_conn, disconn=_disconn)
async def live(send): pass

In [ ]:
#| export
def _broadcast(fragment):
    "Push fragment to every WS client from any thread; drop clients that fail."
    with _clients_lock: targets = list(_clients)
    for loop, send in targets:
        try: fut = asyncio.run_coroutine_threadsafe(send(fragment), loop)
        except Exception: _drop(send); continue
        fut.add_done_callback(lambda f, s=send: (not f.cancelled()) and f.exception() is not None and _drop(s))

def inspector(port=8000, height=520, profile='standard'):
    "Start the in-kernel live inspector panel and return the inline iframe."
    global _server, _profile
    _profile = profile
    if _server is None:
        _server = JupyUvi(app, port=port, daemon=True)
        on_change(lambda: _broadcast(to_xml(_loader(oob=True))))
    return HTMX(port=port, height=height, link=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()